# Уникальные скороговорки датасета

Подход:
1. Транскрибируем **все** WAV-файлы через Whisper.
2. Исправляем ошибки транскрипции через LLM (Ollama).
3. Сохраняем все записи с текстом и контрольными буквами в `data/tongue_twisters.csv`.
4. Считаем статистику уникальных скороговорок.

In [ ]:
import sys
import re
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))
from shared import config
from shared.data_utils import load_audio

CONTROL_LETTERS = list('лрстцчшщ')
print(f'DATA_DIR: {config.DATA_DIR}')
print(f'Контрольные буквы: {CONTROL_LETTERS}')

## 1. Загрузка Whisper

In [ ]:
WHISPER_ID = "openai/whisper-small"
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Загружаем {WHISPER_ID} на {device}...")

processor = WhisperProcessor.from_pretrained(WHISPER_ID)
whisper   = WhisperForConditionalGeneration.from_pretrained(WHISPER_ID).to(device)
whisper.eval()
print("Готово.")

## 2. Вспомогательные функции

In [ ]:
RUSSIAN_LOWER = frozenset(chr(c) for c in list(range(0x0430, 0x0450)) + [0x0451])

def fix_utf8_corrupted_on_windows(s):
    out = []
    i = 0
    while i < len(s):
        if i + 1 >= len(s):
            out.append(s[i]); i += 1; continue
        ord1, ord2 = ord(s[i]), ord(s[i+1])
        if ord2 < 0x80 or ord2 > 0xBF:
            if ord1 == 0x0430 and ord2 == 0x041B:
                byte2 = 0xBB
            else:
                out.append(s[i]); i += 1; continue
        else:
            byte2 = ord2
        first_byte = 0xD1 if ord1 == 0x0431 else (0xD0 if ord1 == 0x0430 else None)
        if first_byte is not None:
            try:
                out.append(bytes([first_byte, byte2]).decode('utf-8')); i += 2; continue
            except UnicodeDecodeError:
                pass
        out.append(s[i]); i += 1
    return ''.join(out)

def get_letter_combo(path):
    """Возвращает frozenset контрольных букв из имени файла."""
    stem = Path(path).stem
    raw  = stem.split('__', 1)[1] if '__' in stem else ''
    decoded = fix_utf8_corrupted_on_windows(raw)
    letters = frozenset(c for c in decoded if c in RUSSIAN_LOWER)
    return letters

def normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def transcribe(path: str) -> str:
    y, _ = load_audio(path, sr=16000)
    inputs = processor(y, sampling_rate=16000, return_tensors="pt")
    input_features = inputs.input_features.to(device)
    # attention_mask нужен явно — pad_token совпадает с eos_token у Whisper
    attention_mask = (input_features != processor.feature_extractor.padding_value).long()
    with torch.no_grad():
        ids = whisper.generate(
            input_features,
            attention_mask=attention_mask,
            language="russian",
            task="transcribe",
            suppress_tokens=[],       # отключаем конфликтующий SuppressTokensLogitsProcessor
        )
    return processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

## 3. Транскрипция всех файлов

In [ ]:
WHISPER_CACHE = config.DATA_DIR / 'cache_whisper.csv'

if WHISPER_CACHE.exists():
    df_transcripts = pd.read_csv(WHISPER_CACHE, encoding='utf-8')
    print(f"Загружено из кэша: {WHISPER_CACHE}")
    print(f"Записей: {len(df_transcripts)}")
else:
    paths_good = sorted(config.GOOD_DIR.glob('*.wav'))
    paths_bad  = sorted(config.BAD_DIR.glob('*.wav'))
    all_paths  = [(str(p), config.CLASS_GOOD) for p in paths_good] + \
                 [(str(p), config.CLASS_BAD)  for p in paths_bad]

    print(f"Всего файлов: {len(all_paths)}")

    records = []
    for i, (path, label) in enumerate(all_paths):
        try:
            raw  = transcribe(path)
            norm = normalize(raw)
        except Exception:
            raw, norm = '', ''
        records.append({
            'path':      path,
            'label':     label,
            'raw_text':  raw,
            'norm_text': norm,
            'combo':     ''.join(sorted(get_letter_combo(path))),
        })
        if (i + 1) % 100 == 0 or (i + 1) == len(all_paths):
            print(f"  {i+1}/{len(all_paths)}")

    df_transcripts = pd.DataFrame(records)

    df_transcripts.to_csv(WHISPER_CACHE, index=False, encoding='utf-8')
    print(f"\nСохранено в кэш: {WHISPER_CACHE}")

print(f"Пустых транскрипций: {(df_transcripts.norm_text == '').sum()}")
df_transcripts.head(3)

## 3а. Коррекция транскрипций через LLM (Ollama)

Прогоняем каждую транскрипцию через локальную модель Ollama —
она исправляет ошибки Whisper, восстанавливает правильный текст скороговорки.

Перед запуском убедись, что Ollama запущена:
```bash
ollama serve
ollama pull qwen2.5   # один раз
```

In [ ]:
import ollama

OLLAMA_MODEL = "qwen2.5"  # можно заменить на llama3.2, gemma3 и др.

SYSTEM_PROMPT = (
    "Ты помощник, который исправляет ошибки автоматической транскрипции русских скороговорок. "
    "Тебе дан текст, распознанный системой Whisper — в нём могут быть опечатки, "
    "пропущенные или перепутанные слова. "
    "Верни ТОЛЬКО исправленный текст скороговорки, без пояснений и кавычек."
)

def correct_text(raw: str) -> str:
    if not raw.strip():
        return ''
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": raw},
        ],
    )
    return response['message']['content'].strip()

# Проверка соединения
try:
    ollama.list()
    print(f"Ollama доступна. Модель: {OLLAMA_MODEL}")
    print(f"Файлов для коррекции: {len(df_transcripts)}")
except Exception as e:
    print(f"Ollama недоступна: {e}")
    print("Запусти: ollama serve && ollama pull qwen2.5")

In [ ]:
OLLAMA_CACHE = config.DATA_DIR / 'cache_ollama.csv'

if OLLAMA_CACHE.exists():
    df_corrected = pd.read_csv(OLLAMA_CACHE, encoding='utf-8')
    df_transcripts['corrected_text'] = df_corrected['corrected_text']
    df_transcripts['corrected_norm'] = df_corrected['corrected_norm']
    print(f"Загружено из кэша: {OLLAMA_CACHE}")
    print(f"Записей: {len(df_transcripts)}")
else:
    corrected = []
    errors = 0
    for i, row in df_transcripts.iterrows():
        try:
            text = correct_text(row['raw_text'])
        except Exception as e:
            text = row['raw_text']
            errors += 1
        corrected.append(text)
        if (i + 1) % 100 == 0 or (i + 1) == len(df_transcripts):
            print(f"  {i+1}/{len(df_transcripts)}  (ошибок API: {errors})")

    df_transcripts['corrected_text'] = corrected
    df_transcripts['corrected_norm'] = df_transcripts['corrected_text'].apply(normalize)

    df_transcripts[['corrected_text', 'corrected_norm']].to_csv(OLLAMA_CACHE, index=False, encoding='utf-8')
    print(f"\nСохранено в кэш: {OLLAMA_CACHE}")
    print(f"Ошибок API: {errors}")

print(f"Пустых после коррекции: {(df_transcripts.corrected_norm == '').sum()}")
df_transcripts[['raw_text', 'corrected_text']].head(5)

## 4. Сборка и сохранение

In [ ]:
# Добавляем контрольные буквы к каждой записи
for letter in CONTROL_LETTERS:
    df_transcripts[letter] = df_transcripts['path'].apply(
        lambda p: int(letter in get_letter_combo(p))
    )

col_order = ['path', 'label', 'raw_text', 'corrected_text', 'corrected_norm'] + CONTROL_LETTERS
df_out = df_transcripts[col_order].copy()

out_path = config.DATA_DIR / 'tongue_twisters.csv'
df_out.to_csv(out_path, index=False, encoding='utf-8')
print(f"Сохранено: {out_path}")
print(f"Всего записей: {len(df_out)}")
df_out.head(5)

## 5. Статистика уникальных скороговорок

In [ ]:
import matplotlib.pyplot as plt

# Уникальность по исправленному нормализованному тексту
n_total        = len(df_out)
n_unique_corr  = df_out['corrected_norm'].nunique()
n_unique_raw   = df_out[df_out['raw_text'] != '']['raw_text'].apply(
                     lambda t: t.lower().strip()).nunique()
n_empty        = (df_out['corrected_norm'] == '').sum()

print("=" * 50)
print(f"Всего записей:                {n_total}")
print(f"Уникальных (raw Whisper):     {n_unique_raw}")
print(f"Уникальных (после LLM):       {n_unique_corr}")
print(f"Пустых транскрипций:          {n_empty}")
print("=" * 50)

# Топ-30 самых частых скороговорок
top = (df_out[df_out['corrected_norm'] != '']
       .groupby('corrected_norm')
       .agg(
           n_total  =('label', 'count'),
           n_good   =('label', lambda x: (x == 0).sum()),
           n_bad    =('label', lambda x: (x == 1).sum()),
           text     =('corrected_text', lambda x: x.mode()[0] if len(x) else ''),
       )
       .reset_index()
       .sort_values('n_total', ascending=False)
       .head(30))

print(f"\nТоп-30 скороговорок по числу записей:")
display(top[['text', 'n_total', 'n_good', 'n_bad']].reset_index(drop=True))

# График распределения частот
counts_all = (df_out[df_out['corrected_norm'] != '']
              .groupby('corrected_norm').size().sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(range(len(top)), top['n_total'], color='steelblue', label='total')
axes[0].bar(range(len(top)), top['n_bad'],   color='coral',     label='bad', alpha=0.8)
axes[0].set_xticks(range(len(top)))
axes[0].set_xticklabels([t[:25] for t in top['text']], rotation=90, fontsize=6)
axes[0].set_ylabel('Количество записей')
axes[0].set_title('Топ-30 скороговорок')
axes[0].legend()

axes[1].hist(counts_all.values, bins=30, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Записей на скороговорку')
axes[1].set_ylabel('Количество скороговорок')
axes[1].set_title(f'Распределение частот ({n_unique_corr} уникальных)')
axes[1].axvline(counts_all.mean(), color='red', linestyle='--',
                label=f'среднее {counts_all.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(exp_dir / 'tongue_twisters_stats.png', dpi=150, bbox_inches='tight')
plt.show()